**Nota:** Este cuaderno es una copia de la primera parte de [01_exploración_inicial.ipynb](01_exploración_inicial.ipynb) (hasta la eliminación de `url`, `emp_title`, `title` y `zip_code`), pero carga los datos con la base común de [`src/data.py`](../src/data.py). Por eso:

- La población y el target son los de la base común: se excluye `Default` (1,348,059 préstamos).
- La partición es 64/16/20 (entrenamiento/validación/prueba) con `SEED = 42`, en lugar de 80/20.
- Las columnas con leakage, las posteriores a la decisión y los identificadores (`id`, `member_id`, `url`) ya vienen excluidas desde la carga.
- `issue_d` no entra en `X`, porque `proposal.md` no la admite como predictor; queda en `entrenamiento`, `validacion` y `prueba` para análisis por cohorte.

Al terminar quedan disponibles `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test` con las mismas columnas, para continuar con los issues siguientes.

## Definición del Problema:

Las instituciones de crédito enfrentan pérdidas financieras significativas cuando los prestatarios no pagan sus créditos (default). El objetivo de este proyecto es construir un modelo de clasificación supervisada capaz de predecir la probabilidad de que un cliente caiga en mora grave (Charged Off) al momento de solicitar un préstamo.
- Unidad de predicción: Una solicitud individual de préstamo.
- Variable objetivo: loan_status binarizada (1 = Charged Off, 0 = Fully Paid).
- Decisión de negocio: Aprobar, rechazar o ajustar la tasa de interés según el riesgo estimado antes de otorgar el dinero

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from src.data import (
    IDENTIFICADORES, LEAKAGE_COLS, POST_DECISION_COLS,
    cargar_prestamos, particionar, ruta_datos, separar_X_y,
)

## Realizamos la carga de los datos

In [2]:
# Préstamos resueltos, con target y partición (base común de src/data.py).
# La primera vez lee el CSV y guarda una caché Parquet; después tarda unos segundos.
df_work = cargar_prestamos()

print(f"Filas en df_work: {df_work.shape[0]:,}")
print(f"Columnas en df_work: {df_work.shape[1]}")

print("¿Existe 'target' en df_work?:", 'target' in df_work.columns)

Filas en df_work: 1,348,059
Columnas en df_work: 104
¿Existe 'target' en df_work?: True


## Análisis del tipo de datos

### Columnas con riesgo de leakage

In [3]:
# Las listas viven en src/data.py y ya se excluyen al cargar.
# Aquí solo se comprueba que ninguna quedó en los datos.
excluidas = [*LEAKAGE_COLS, *POST_DECISION_COLS, *IDENTIFICADORES]
presentes = [c for c in excluidas if c in df_work.columns]
print(f"Columnas excluidas por src/data.py: {len(excluidas)}")
print(f"Columnas excluidas que siguen presentes: {presentes}")

Columnas excluidas por src/data.py: 48
Columnas excluidas que siguen presentes: []


In [4]:
print(f"Filas totales: {df_work.shape[0]:,}")
print(f"Columnas totales: {df_work.shape[1]}")

# Tipos de datos
print("\n=== TIPOS DE DATOS ===")
print(df_work.dtypes.value_counts())

# Calcular % de nulos por columna
missing = (df_work.isnull().sum() / len(df_work)) * 100
missing_df = missing[missing > 0].sort_values(ascending=False).reset_index()
missing_df.columns = ['Variable', 'Porcentaje_Nulos']

# Mostrar las 6 variables con más faltantes
missing_df.head(6)

Filas totales: 1,348,059
Columnas totales: 104

=== TIPOS DE DATOS ===
float64           86
object            15
datetime64[ns]     1
int8               1
category           1
Name: count, dtype: int64


,Variable,Porcentaje_Nulos
0,sec_app_mths_since_last_major_derog,99.507069
1,sec_app_revol_util,98.642344
2,revol_bal_joint,98.618087
3,sec_app_open_act_il,98.618013
4,sec_app_inq_last_6mths,98.618013
5,sec_app_collections_12_mths_ex_med,98.618013


In [5]:
# La base común reemplaza loan_status por target, así que la distribución
# original se lee del CSV (solo esa columna; tarda alrededor de un minuto).
pd.read_csv(ruta_datos(), usecols=['loan_status'])["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

### Definición de la variable objetivo

Queremos predecir si un préstamo termina en **Charged Off**, es decir, si el prestamista lo da por perdido. Para ello solo se va a usar préstamos cuyo resultado final ya se conoce.

| `loan_status` | Etiqueta | Motivo |
|---|---|---|
| Charged Off | **1** | Es el evento que queremos predecir. |
| Does not meet the credit policy. Status:Charged Off | **1** | Es un Charged Off. El prefijo solo indica que se otorgó con una política de crédito antigua. |
| Fully Paid | **0** | El préstamo se pagó por completo. |
| Does not meet the credit policy. Status:Fully Paid | **0** | Es un Fully Paid otorgado con la política de crédito antigua. |
| Default | Se excluye | Son muy pocos casos (40) y no es un resultado definitivo. |
| Current, In Grace Period, Late (16-30 days), Late (31-120 days) | Se excluye | El préstamo sigue activo y todavía no sabemos cómo terminará. |
| `NaN` | Se excluye | No tiene estado. |

### ¿Por qué no se etiquetan los préstamos activos como 0?

Ponerles 0 sería afirmar que *no* terminaron en Charged Off, y eso todavía no se sabe:

- Una parte de los préstamos **Current** todavía terminará en Charged Off, así que etiquetarlos como 0 introduce etiquetas falsas.
- Los préstamos **Late** e **In Grace Period** ya tienen pagos atrasados. Una fracción importante de ellos terminará en Charged Off, así que etiquetarlos como 0 sería especialmente engañoso.
- Con esas etiquetas falsas, el modelo aprendería que perfiles riesgosos "no fallan". Subestimaría el riesgo, y sus métricas tampoco serían fiables.

In [6]:
# La tabla anterior ya está aplicada en src/data.py: df_work solo contiene
# préstamos resueltos y target vale 1 para Charged Off y 0 para Fully Paid.
print(df_work['target'].value_counts())
print(f"Tasa de Charged Off: {df_work['target'].mean():.4f}")

target
0    1078739
1     269320
Name: count, dtype: int64
Tasa de Charged Off: 0.1998


# Separación de data de entrenamiento, validación y prueba

In [7]:
# Partición común 64/16/20, estratificada por target con SEED = 42.
entrenamiento, validacion, prueba = particionar(df_work)

# loan_amnt se incluye como en 01 (incluir_condicionales=True); issue_d queda fuera.
X_train, y_train = separar_X_y(entrenamiento, incluir_condicionales=True)
X_val, y_val = separar_X_y(validacion, incluir_condicionales=True)
X_test, y_test = separar_X_y(prueba, incluir_condicionales=True)

for nombre, X_, y_ in [('Entrenamiento', X_train, y_train),
                       ('Validación', X_val, y_val),
                       ('Prueba', X_test, y_test)]:
    print(f"{nombre}: {len(X_):,} filas, {X_.shape[1]} columnas, Charged Off {y_.mean():.2%}")

Entrenamiento: 862,757 filas, 101 columnas, Charged Off 19.98%
Validación: 215,690 filas, 101 columnas, Charged Off 19.98%
Prueba: 269,612 filas, 101 columnas, Charged Off 19.98%


# Análisis de features candidatos

## Eliminiación de valores nulos > 90 %

In [8]:
missing = (X_train.isnull().sum() / len(X_train)) * 100
missing_df = missing[missing > 0].sort_values(ascending=False).reset_index()
missing_df.columns = ['Variable', 'Porcentaje_Nulos']
mayor90=missing_df[missing_df['Porcentaje_Nulos'] > 90]["Variable"].to_list()
X_train=X_train.drop(columns=mayor90)

In [9]:
missing_df = missing_df[(missing_df['Porcentaje_Nulos'] > 0) &
                        (missing_df['Porcentaje_Nulos'] < 90)]
cols = missing_df['Variable'].tolist()

## Detectamos columnas con valores similares para cada fila y los qu tienen muchos valores diferentes

In [10]:
resumen = pd.DataFrame({
      'nulos_%': X_train.isna().mean() * 100,
      'unicos': X_train.nunique(),
      'tipo': X_train.dtypes,
})
# Columnas constantes o casi constantes
casi_constantes = [c for c in X_train.columns if X_train[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.99]

In [11]:
X_train.drop(columns=casi_constantes, inplace=True)
resumen.drop(index=casi_constantes, inplace=True)

In [12]:
resumen[resumen['tipo'] == 'object'].sort_values('unicos', ascending=False)

,nulos_%,unicos,tipo
emp_title,6.381635,266064,object
title,1.236849,43145,object
zip_code,0.000116,934,object
earliest_cr_line,0.002318,732,object
addr_state,0.000000,51,object
purpose,0.000000,14,object
emp_length,5.833045,11,object
home_ownership,0.000000,6,object
verification_status,0.000000,3,object
term,0.000000,2,object


De estos valores podemos eliminar de manera segura emp_title, title y zip code (`url` ya se excluye en `src/data.py`)

In [13]:
X_train.drop(columns=["emp_title","title","zip_code"], inplace=True)
resumen.drop(index=["emp_title","title","zip_code"], inplace=True)

## Mismas columnas en validación y prueba

Las columnas eliminadas se decidieron solo con entrenamiento; validación y prueba se recortan a las mismas columnas, sin volver a calcular nada con ellas.

In [14]:
X_val = X_val[X_train.columns]
X_test = X_test[X_train.columns]

print(f"Columnas finales: {X_train.shape[1]}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Columnas finales: 76
X_train: (862757, 76) | X_val: (215690, 76) | X_test: (269612, 76)
